## This tends to solve Task5 , 6

Task5 and 6 brings revenue and cost into the scope. In terms of decision making, there is a balance between opeartional cost for timely OTD and lost sales in demand

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
import warnings
warnings.filterwarnings('ignore')

## Strategy

**Objective:** Maximize marginal profit = Revenue - Shipping Cost

**Approach:** Brute force optimization over all lead time combinations
- 3 market types × 6 lead time options = 6³ = 216 scenarios
- Each scenario assigns one OTD lead time (1-5+ days) to each market type

**Revenue Model:**
- Converted demand = Total demand × Conversion rate (market, lead_time)
- Revenue = Converted demand × Unit price ($50 assumed)

**Cost Model:**
- Shipping cost varies by distance zone (8 zones) and lead time
- Total cost = Σ (allocated_demand × conversion_rate × unit_cost)

**Output:** Optimal lead time configuration and sensitivity analysis

In [5]:
# Load OTD data
otd_sales = pd.read_csv('data/OTD_Sales.csv')
otd_cost = pd.read_csv('data/OTD_Cost.csv')

# Load allocation from Task 3/4 (use 15-FC scenario)
alloc_15fc = pd.read_csv('data/zip3_fc_allocation_15fc.csv') # Or reload if needed

# Load base data
demand_df = pd.read_csv('data/simulation_results.csv')
zip3_market_df = pd.read_csv('data/zip3_market.csv').rename(columns={'Market': 'market_type'})

print("Data loaded successfully")
print(f"OTD Sales shape: {otd_sales.shape}")
print(f"OTD Cost shape: {otd_cost.shape}")
print(f"Allocation shape: {alloc_15fc.shape}")

Data loaded successfully
OTD Sales shape: (3, 7)
OTD Cost shape: (8, 7)
Allocation shape: (1376, 6)


In [6]:
# Distance zone mapping
def get_distance_zone(distance):
    if distance < 50: return 1
    elif distance < 150: return 2
    elif distance < 300: return 3
    elif distance < 600: return 4
    elif distance < 1000: return 5
    elif distance < 1400: return 6
    elif distance < 1800: return 7
    else: return 8

# Prepare base dataset: ZIP3-FC-Market with demand and distance
base_data = demand_df.merge(alloc_15fc[['ZIP3', 'FC', 'allocation_pct', 'distance']], on='ZIP3')
base_data['distance_zone'] = base_data['distance'].apply(get_distance_zone)
base_data = base_data.merge(zip3_market_df, on='ZIP3')
base_data['allocated_demand'] = base_data['Mean_Demand'] * base_data['allocation_pct']

# Aggregate across days (annual demand)
aggregated = base_data.groupby(['ZIP3', 'FC', 'market_type', 'distance_zone']).agg({
    'allocated_demand': 'sum'
}).reset_index()

print(f"Base data shape: {aggregated.shape}")
print(f"Market types: {aggregated['market_type'].unique()}")
print(f"Distance zones: {sorted(aggregated['distance_zone'].unique())}")

Base data shape: (1288, 5)
Market types: ['Primary' 'Secondary' 'Tertiary']
Distance zones: [1, 2, 3, 4, 5]


In [15]:
# Setup conversion rate and cost lookup tables
# OTD_Sales.csv: Market × Lead Time → Conversion Rate
# OTD_Cost.csv: Zone × Lead Time → Cost per unit

UNIT_PRICE = 3000  
LEAD_TIMES = [1, 2, 3, 4, 5, '5+']
MARKETS = ['Primary', 'Secondary', 'Tertiary']

# Create lookup dictionaries
conversion_lookup = {}
for _, row in otd_sales.iterrows():
    market = row['Market']
    for lt in LEAD_TIMES:
        conversion_lookup[(market, str(lt))] = row[str(lt)]

# preprocess or there will be KeyError
otd_cost = otd_cost.rename(columns={'Zone (miles)': 'Zone'})
otd_cost['Zone'] = list(range(1, len(otd_cost)+1))


cost_lookup = {}
for _, row in otd_cost.iterrows():
    zone = row['Zone']
    for lt in LEAD_TIMES:
        cost_lookup[(zone, str(lt))] = row[str(lt)]

print("Lookup tables created")
print(f"Sample conversion: Primary-5day = {conversion_lookup[('Primary', '5')]}")
print(f"Sample cost: Zone1-1day = ${cost_lookup[(1, '1')]}")

Lookup tables created
Sample conversion: Primary-5day = 40%
Sample cost: Zone1-1day = $$607


In [16]:
# Brute force optimization function
def calculate_marginal_profit(lead_time_config, data):
    """
    lead_time_config: dict {market_type: lead_time}
    Returns: (total_revenue, total_cost, marginal_profit)
    """
    total_revenue = 0
    total_cost = 0
    
    for _, row in data.iterrows():
        market = row['market_type']
        zone = row['distance_zone']
        demand = row['allocated_demand']
        
        lead_time = str(lead_time_config[market])
        conversion_rate = conversion_lookup.get((market, lead_time), 0)
        unit_cost = cost_lookup.get((zone, lead_time), 0)
        
        converted_demand = demand * conversion_rate
        revenue = converted_demand * UNIT_PRICE
        cost = converted_demand * unit_cost
        
        total_revenue += revenue
        total_cost += cost
    
    marginal_profit = total_revenue - total_cost
    return total_revenue, total_cost, marginal_profit

# Generate all combinations
lead_time_options = [1, 2, 3, 4, 5, '5+']
all_combinations = list(product(lead_time_options, repeat=3))

print(f"Total scenarios to evaluate: {len(all_combinations)}")

Total scenarios to evaluate: 216


In [17]:
# Run optimization
results = []

for combo in all_combinations:
    config = {
        'Primary': combo[0],
        'Secondary': combo[1],
        'Tertiary': combo[2]
    }
    
    revenue, cost, profit = calculate_marginal_profit(config, aggregated)
    
    results.append({
        'Primary_LT': combo[0],
        'Secondary_LT': combo[1],
        'Tertiary_LT': combo[2],
        'Total_Revenue': revenue,
        'Total_Cost': cost,
        'Marginal_Profit': profit
    })

results_df = pd.DataFrame(results).sort_values('Marginal_Profit', ascending=False)

# Optimal solution
optimal = results_df.iloc[0]
print("="*80)
print("OPTIMAL LEAD TIME CONFIGURATION")
print("="*80)
print(f"Primary Market:   {optimal['Primary_LT']} days")
print(f"Secondary Market: {optimal['Secondary_LT']} days")
print(f"Tertiary Market:  {optimal['Tertiary_LT']} days")
print(f"\nTotal Revenue:     ${optimal['Total_Revenue']:,.2f}")
print(f"Total Cost:        ${optimal['Total_Cost']:,.2f}")
print(f"Marginal Profit:   ${optimal['Marginal_Profit']:,.2f}")
print(f"\nTop 5 configurations:")
print(results_df.head(10)[['Primary_LT', 'Secondary_LT', 'Tertiary_LT', 'Marginal_Profit']])

TypeError: can't multiply sequence by non-int of type 'float'

In [ ]:
# Sensitivity analysis visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, market in enumerate(['Primary', 'Secondary', 'Tertiary']):
    col_name = f'{market}_LT'
    market_data = results_df.groupby(col_name)['Marginal_Profit'].mean().sort_index()
    
    axes[idx].bar(range(len(market_data)), market_data.values, color='steelblue', alpha=0.7)
    axes[idx].set_xticks(range(len(market_data)))
    axes[idx].set_xticklabels(market_data.index)
    axes[idx].set_xlabel('Lead Time (days)', fontsize=11)
    axes[idx].set_ylabel('Avg Marginal Profit ($)', fontsize=11)
    axes[idx].set_title(f'{market} Market Sensitivity', fontsize=12, fontweight='bold')
    axes[idx].grid(axis='y', alpha=0.3)
    
    # Highlight optimal
    optimal_lt = optimal[col_name]
    optimal_idx = list(market_data.index).index(optimal_lt)
    axes[idx].bar(optimal_idx, market_data.values[optimal_idx], color='crimson', alpha=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# 3D Profit landscape (fixing Tertiary at optimal)
from mpl_toolkits.mplot3d import Axes3D

tertiary_optimal = optimal['Tertiary_LT']
filtered = results_df[results_df['Tertiary_LT'] == tertiary_optimal].copy()

# Create meshgrid
primary_vals = sorted(filtered['Primary_LT'].unique())
secondary_vals = sorted(filtered['Secondary_LT'].unique())
profit_matrix = filtered.pivot(index='Primary_LT', columns='Secondary_LT', values='Marginal_Profit')

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

X, Y = np.meshgrid(range(len(secondary_vals)), range(len(primary_vals)))
Z = profit_matrix.values

surf = ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8)
ax.set_xlabel('Secondary Market LT', fontsize=10)
ax.set_ylabel('Primary Market LT', fontsize=10)
ax.set_zlabel('Marginal Profit ($)', fontsize=10)
ax.set_title(f'Profit Landscape (Tertiary fixed at {tertiary_optimal} days)', fontsize=12, fontweight='bold')
ax.set_xticks(range(len(secondary_vals)))
ax.set_xticklabels(secondary_vals)
ax.set_yticks(range(len(primary_vals)))
ax.set_yticklabels(primary_vals)

plt.colorbar(surf, ax=ax, shrink=0.5)
plt.tight_layout()
plt.show()

## Key Insights

**Trade-off Analysis:**
- Shorter lead times → Higher conversion rates → More revenue
- Longer lead times → Lower shipping costs → Higher margin per unit
- Optimal balance depends on market characteristics and distance distribution

**Market Behavior:**
- **Primary markets**: More price-sensitive, demand fast delivery
- **Secondary markets**: Moderate sensitivity
- **Tertiary markets**: More cost-conscious, accept longer lead times

**Strategic Recommendation:** Differentiated service levels by market type maximize overall profitability